# Banking Data Ingest — Bronze Layer

**Task 1 of the `banking_data_workflow` Lakeflow job.**

This notebook reads the raw banking CSVs from the lab volume (`/Volumes/databricks_labs/default/lab_data/week2/banking/`) with an explicit, enforced schema and creates the `bank_master_data_bronze` table. It then publishes the bronze row count (`bronze_count`) as a task value so downstream tasks (`borrower_details`, `loan_details`) can validate it.

* Downstream: `borrower_details` and `loan_details` (silver layer)
* Output table: `bank_master_data_bronze` (~5,000 rows)
* Task value published: `bronze_count`

In [0]:
SELECT COUNT(*) FROM bank_master_data_bronze

In [0]:
-- Ingest raw banking CSVs into the bronze table with an explicit, enforced schema
CREATE OR REPLACE TABLE bank_master_data_bronze
AS
select * from read_files(
    "/Volumes/databricks_labs/default/lab_data/week2/banking/",
    format => "csv",
    header => "true",
    enforceSchema => "true",
    schema => "
        ID INT,
        Age INT,
        Experience INT,
        Income DOUBLE,
        ZIP_Code STRING,
        Family INT,
        CCAvg DOUBLE,
        Education INT,
        Mortgage DOUBLE,
        Personal_Loan INT,
        Securities_Account INT,
        CD_Account INT,
        Online INT,
        CreditCard INT
    "
);

In [0]:
select * from bank_master_data_bronze;

In [0]:
%python
# Publish the bronze row count as a task value for downstream job tasks
dbutils.jobs.taskValues.set(
    key="bronze_count",
    value=spark.table("bank_master_data_bronze").count()
)